In [7]:
import pandas as pd
import geopandas as gpd

# 1. 读取全英 ward 地图和对照表
# 请将路径替换为你实际的文件路径
uk_wards = gpd.read_file('data/Wards_December_2011_GCB.geojson')
lookup_table = pd.read_csv('data/Wards_and_Local_Authorities_(2011)_to_Wards_and_Local_Authorities_(2022)_Lookup_in_England_and_Wales.csv')

lookup_table = lookup_table.drop_duplicates(subset=['WD11CD'])

# 2. 将对照表合并到地图数据上 (类似 Excel 的 VLOOKUP)
# 注意大小写匹配：左边地图是 wd11cd，右边对照表是 WD11CD
merged_wards = uk_wards.merge(lookup_table, left_on='wd11cd', right_on='WD11CD', how='left')

# 3. 黄金筛选技巧：选出伦敦的 Wards
# 大伦敦地区所有的自治市代码 (LAD11CD) 都是以 'E09' 开头的！
london_wards = merged_wards[merged_wards['LAD11CD'].str.startswith('E09', na=False)]

# 4. (可选) 清理数据
# 合并后会多出很多重复或不需要的列，可以把它们删掉，保持 DataFrame 清爽
columns_to_drop = ['WD11CD', 'WD11NM', 'WD22CD', 'WD22NM', 'LAD22CD', 'LAD22NM', 'wd11cdo', 'wd11n']
london_wards = london_wards.drop(columns=columns_to_drop)

# 打印看看结果和行数
print(f"筛选出了 {len(london_wards)} 个伦敦的 Wards")
display(london_wards.head())


筛选出了 649 个伦敦的 Wards


,OBJECTID,wd11cd,wd11cdo,wd11nm,wd11nmw,GlobalID,geometry,LAD11CD,LAD11NM,ObjectId
0,1,E05000001,00AAFA,Aldersgate,,64cb20c8-32b0-4f08-a116-95ffd53af4eb,"POLYGON ((-0.09537 51.51749, -0.09602 51.51608...",E09000001,City of London,10866
1,2,E05000002,00AAFB,Aldgate,,97f04007-2afc-4c5f-97cf-ef203a026f55,"POLYGON ((-0.07783 51.51581, -0.07575 51.51465...",E09000001,City of London,11059
2,3,E05000003,00AAFC,Bassishaw,,c0ea55b2-fe08-42b8-9c1d-3ae222d60654,"POLYGON ((-0.09107 51.51805, -0.08948 51.5175,...",E09000001,City of London,11060
3,4,E05000004,00AAFD,Billingsgate,,e3c0bcd5-21cb-4bd7-88af-d4701435b266,"POLYGON ((-0.08031 51.50801, -0.0815 51.50839,...",E09000001,City of London,10853
4,5,E05000005,00AAFE,Bishopsgate,,7274a237-267f-4592-9dd4-4d0423de9a10,"POLYGON ((-0.07845 51.52151, -0.0794 51.51885,...",E09000001,City of London,10854


In [8]:
# 5. 保存这个只有伦敦的地图，以后直接用这个小文件！
london_wards.to_file('london_wards_2011.geojson', driver='GeoJSON')